In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

RAW_DATA_DIR = Path("C:\\Users\\aabre\\Documents\\data-science-projects\\nba-player-archetype-clustering\\data\\raw\\2025_26")
PROCESSED_DATA_DIR = Path("C:\\Users\\aabre\\Documents\\data-science-projects\\nba-player-archetype-clustering\\data\\processed\\2025_26")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR

WindowsPath('C:/Users/aabre/Documents/data-science-projects/nba-player-archetype-clustering/data/processed/2025_26')

In [2]:
datasets = {}
for file in RAW_DATA_DIR.glob("*.csv"):
    dataset_name = file.stem
    if dataset_name.startswith("0"):  # Skips our inventory/summary files
        continue
    datasets[dataset_name] = pd.read_csv(file)

In [3]:
for name, df in datasets.items():
    if "PLAYER_ID" in df.columns:
        print(name, df["PLAYER_ID"].duplicated().sum())

advanced_stats 0
base_stats 0
catchshoot_stats 0
defense_stats 0
drives_stats 0
efficiency_stats 0
elbowtouch_stats 0
painttouch_stats 0
passing_stats 0
possessions_stats 0
posttouch_stats 0
pullupshot_stats 0
rebounding_stats 0
scoring_splits 0
speeddistance_stats 0
usage_stats 0


In [4]:
datasets["base_stats"]["MIN"].describe()

count     582.000000
mean     1019.758648
std       771.980982
min         2.666667
25%       293.239583
50%       940.295833
75%      1609.389167
max      2953.150000
Name: MIN, dtype: float64

In [3]:
# 250 total minutes played will be our chosen threshold for eligibility.
eligible_players = datasets["base_stats"][datasets["base_stats"]["MIN"] >= 250].copy()
eligible_players.shape

(450, 69)

In [4]:
players = eligible_players[["PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "AGE", "GP", "MIN"]].copy()
print(players.head())
players["PLAYER_ID"].nunique() == players.shape[0]

   PLAYER_ID    PLAYER_NAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP  \
1    1631260       AJ Green  1610612749               MIL  26.0  78   
2    1642358     AJ Johnson  1610612742               DAL  21.0  48   
3     203932   Aaron Gordon  1610612743               DEN  30.0  36   
4    1628988  Aaron Holiday  1610612745               HOU  29.0  57   
5    1630174  Aaron Nesmith  1610612754               IND  26.0  45   

           MIN  
1  2269.933333  
2   453.433333  
3  1004.958333  
4   780.731667  
5  1334.950000  


True

In [6]:
player_check = []
player_ids = set(players["PLAYER_ID"])
for name, df in datasets.items():
    if "PLAYER_ID" not in df.columns:
        continue
    dataset_ids = set(df["PLAYER_ID"])
    players_available = len(player_ids.intersection(dataset_ids))
    player_check.append({
        "dataset": name, 
        "players_available": players_available, 
        "total_players": len(player_ids), 
        "coverage": (players_available / len(player_ids) * 100)
        })
player_check_df = pd.DataFrame(player_check)
player_check_df

,dataset,players_available,total_players,coverage
0,advanced_stats,450,450,100.0
1,base_stats,450,450,100.0
2,catchshoot_stats,450,450,100.0
3,defense_stats,450,450,100.0
4,drives_stats,450,450,100.0
5,efficiency_stats,450,450,100.0
6,elbowtouch_stats,450,450,100.0
7,painttouch_stats,450,450,100.0
8,passing_stats,450,450,100.0
9,possessions_stats,450,450,100.0


In [8]:
minute_summary = {}
for name, df in datasets.items():
    if "MIN" in df.columns:
        minute_check = players[["PLAYER_ID", "PLAYER_NAME", "MIN"]].merge(
            df[["PLAYER_ID", "MIN"]], on="PLAYER_ID", how="left", suffixes=("", f"_{name}"))
        minute_check["MIN_diff"] = (minute_check[f"MIN_{name}"] / minute_check["MIN"])
        minute_summary[name] = minute_check["MIN_diff"].describe()
minute_summary_df = pd.DataFrame(minute_summary).T
minute_summary_df

,count,mean,std,min,25%,50%,75%,max
advanced_stats,450.0,0.021341,0.013102,0.012172,0.014297,0.016913,0.022718,0.090928
base_stats,450.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
catchshoot_stats,450.0,1.000026,0.000436,0.998264,0.999807,1.000014,1.000204,1.001711
defense_stats,450.0,1.000023,0.000491,0.995277,0.999814,1.000023,1.000224,1.001711
drives_stats,450.0,1.000023,0.000491,0.995277,0.999814,1.000023,1.000224,1.001711
efficiency_stats,450.0,1.000023,0.000491,0.995277,0.999814,1.000023,1.000224,1.001711
elbowtouch_stats,450.0,1.000023,0.000491,0.995277,0.999814,1.000023,1.000224,1.001711
painttouch_stats,450.0,1.000023,0.000491,0.995277,0.999814,1.000023,1.000224,1.001711
passing_stats,450.0,1.000023,0.000491,0.995277,0.999814,1.000023,1.000224,1.001711
possessions_stats,450.0,1.000023,0.000491,0.995277,0.999814,1.000023,1.000224,1.001711


In [5]:
def feature_finder(*terms):
    """
    Searches for columns in the datasets that contain any of the specified terms.
    
    Args:
        *terms: Variable length argument list of strings to search for in column names.
        
    Returns:
        A dictionary where keys are dataset names and values are lists of matching column names.
    """
    terms = [term.lower() for term in terms]
    matching_features = {}
    for name, df in datasets.items():
        for col in df.columns:
            if any(term in col.lower() for term in terms):
                if name not in matching_features:
                    matching_features[name] = []
                matching_features[name].append(col)
    return matching_features

In [11]:
feature_finder("poss")


{'advanced_stats': ['POSS'], 'possessions_stats': ['TIME_OF_POSS']}

In [10]:
feature_finder("pass", "ast")


{'advanced_stats': ['AST_PCT',
  'AST_TO',
  'AST_RATIO',
  'AST_PCT_RANK',
  'AST_TO_RANK',
  'AST_RATIO_RANK'],
 'base_stats': ['AST', 'AST_RANK'],
 'drives_stats': ['DRIVE_PASSES',
  'DRIVE_PASSES_PCT',
  'DRIVE_AST',
  'DRIVE_AST_PCT'],
 'elbowtouch_stats': ['ELBOW_TOUCH_PASSES',
  'ELBOW_TOUCH_PASSES_PCT',
  'ELBOW_TOUCH_AST',
  'ELBOW_TOUCH_AST_PCT'],
 'painttouch_stats': ['PAINT_TOUCH_PASSES',
  'PAINT_TOUCH_PASSES_PCT',
  'PAINT_TOUCH_AST',
  'PAINT_TOUCH_AST_PCT'],
 'passing_stats': ['PASSES_MADE',
  'PASSES_RECEIVED',
  'AST',
  'FT_AST',
  'SECONDARY_AST',
  'POTENTIAL_AST',
  'AST_PTS_CREATED',
  'AST_ADJ',
  'AST_TO_PASS_PCT',
  'AST_TO_PASS_PCT_ADJ'],
 'posttouch_stats': ['POST_TOUCH_PASSES',
  'POST_TOUCH_PASSES_PCT',
  'POST_TOUCH_AST',
  'POST_TOUCH_AST_PCT'],
 'scoring_splits': ['PCT_AST_2PM',
  'PCT_UAST_2PM',
  'PCT_AST_3PM',
  'PCT_UAST_3PM',
  'PCT_AST_FGM',
  'PCT_UAST_FGM',
  'PCT_AST_2PM_RANK',
  'PCT_UAST_2PM_RANK',
  'PCT_AST_3PM_RANK',
  'PCT_UAST_3PM_RANK',

In [12]:
feature_finder("shoot", "shot", "fg", "3pt", "ft")

{'advanced_stats': ['EFG_PCT',
  'FGM',
  'FGA',
  'FGM_PG',
  'FGA_PG',
  'FG_PCT',
  'EFG_PCT_RANK',
  'FGM_RANK',
  'FGA_RANK',
  'FGM_PG_RANK',
  'FGA_PG_RANK',
  'FG_PCT_RANK'],
 'base_stats': ['FGM',
  'FGA',
  'FG_PCT',
  'FG3M',
  'FG3A',
  'FG3_PCT',
  'FTM',
  'FTA',
  'FT_PCT',
  'FGM_RANK',
  'FGA_RANK',
  'FG_PCT_RANK',
  'FG3M_RANK',
  'FG3A_RANK',
  'FG3_PCT_RANK',
  'FTM_RANK',
  'FTA_RANK',
  'FT_PCT_RANK'],
 'catchshoot_stats': ['CATCH_SHOOT_FGM',
  'CATCH_SHOOT_FGA',
  'CATCH_SHOOT_FG_PCT',
  'CATCH_SHOOT_PTS',
  'CATCH_SHOOT_FG3M',
  'CATCH_SHOOT_FG3A',
  'CATCH_SHOOT_FG3_PCT',
  'CATCH_SHOOT_EFG_PCT'],
 'defense_stats': ['DEF_RIM_FGM', 'DEF_RIM_FGA', 'DEF_RIM_FG_PCT'],
 'drives_stats': ['DRIVE_FGM',
  'DRIVE_FGA',
  'DRIVE_FG_PCT',
  'DRIVE_FTM',
  'DRIVE_FTA',
  'DRIVE_FT_PCT'],
 'efficiency_stats': ['DRIVE_FG_PCT',
  'CATCH_SHOOT_PTS',
  'CATCH_SHOOT_FG_PCT',
  'PULL_UP_FG_PCT',
  'PAINT_TOUCH_FG_PCT',
  'POST_TOUCH_FG_PCT',
  'ELBOW_TOUCH_FG_PCT',
  'EFF_FG_PCT'

In [14]:
feature_finder("def", "loose", "blk", "stl")


{'advanced_stats': ['E_DEF_RATING',
  'DEF_RATING',
  'sp_work_DEF_RATING',
  'E_DEF_RATING_RANK',
  'DEF_RATING_RANK',
  'sp_work_DEF_RATING_RANK'],
 'base_stats': ['STL', 'BLK', 'BLKA', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK'],
 'defense_stats': ['STL',
  'BLK',
  'DEF_RIM_FGM',
  'DEF_RIM_FGA',
  'DEF_RIM_FG_PCT'],
 'rebounding_stats': ['OREB_CHANCE_DEFER',
  'DREB_CHANCE_DEFER',
  'REB_CHANCE_DEFER'],
 'speeddistance_stats': ['DIST_MILES_DEF', 'AVG_SPEED_DEF'],
 'usage_stats': ['PCT_STL',
  'PCT_BLK',
  'PCT_BLKA',
  'PCT_STL_RANK',
  'PCT_BLK_RANK',
  'PCT_BLKA_RANK']}

In [15]:
feature_finder("reb")


{'advanced_stats': ['OREB_PCT',
  'DREB_PCT',
  'REB_PCT',
  'OREB_PCT_RANK',
  'DREB_PCT_RANK',
  'REB_PCT_RANK'],
 'base_stats': ['OREB', 'DREB', 'REB', 'OREB_RANK', 'DREB_RANK', 'REB_RANK'],
 'defense_stats': ['DREB'],
 'rebounding_stats': ['OREB',
  'OREB_CONTEST',
  'OREB_UNCONTEST',
  'OREB_CONTEST_PCT',
  'OREB_CHANCES',
  'OREB_CHANCE_PCT',
  'OREB_CHANCE_DEFER',
  'OREB_CHANCE_PCT_ADJ',
  'AVG_OREB_DIST',
  'DREB',
  'DREB_CONTEST',
  'DREB_UNCONTEST',
  'DREB_CONTEST_PCT',
  'DREB_CHANCES',
  'DREB_CHANCE_PCT',
  'DREB_CHANCE_DEFER',
  'DREB_CHANCE_PCT_ADJ',
  'AVG_DREB_DIST',
  'REB',
  'REB_CONTEST',
  'REB_UNCONTEST',
  'REB_CONTEST_PCT',
  'REB_CHANCES',
  'REB_CHANCE_PCT',
  'REB_CHANCE_DEFER',
  'REB_CHANCE_PCT_ADJ',
  'AVG_REB_DIST'],
 'usage_stats': ['PCT_OREB',
  'PCT_DREB',
  'PCT_REB',
  'PCT_OREB_RANK',
  'PCT_DREB_RANK',
  'PCT_REB_RANK']}

In [16]:
feature_finder("drive", "drib", "ball")


{'drives_stats': ['DRIVES',
  'DRIVE_FGM',
  'DRIVE_FGA',
  'DRIVE_FG_PCT',
  'DRIVE_FTM',
  'DRIVE_FTA',
  'DRIVE_FT_PCT',
  'DRIVE_PTS',
  'DRIVE_PTS_PCT',
  'DRIVE_PASSES',
  'DRIVE_PASSES_PCT',
  'DRIVE_AST',
  'DRIVE_AST_PCT',
  'DRIVE_TOV',
  'DRIVE_TOV_PCT',
  'DRIVE_PF',
  'DRIVE_PF_PCT'],
 'efficiency_stats': ['DRIVE_PTS', 'DRIVE_FG_PCT'],
 'possessions_stats': ['AVG_DRIB_PER_TOUCH']}

In [17]:
feature_finder("usg")


{'advanced_stats': ['USG_PCT', 'E_USG_PCT', 'USG_PCT_RANK', 'E_USG_PCT_RANK'],
 'usage_stats': ['USG_PCT', 'USG_PCT_RANK']}

In [18]:
feature_finder("pace", "speed", "distance")


{'advanced_stats': ['E_PACE',
  'PACE',
  'PACE_PER40',
  'sp_work_PACE',
  'E_PACE_RANK',
  'PACE_RANK',
  'sp_work_PACE_RANK'],
 'speeddistance_stats': ['AVG_SPEED', 'AVG_SPEED_OFF', 'AVG_SPEED_DEF']}

In [19]:
feature_finder("touch")


{'efficiency_stats': ['PAINT_TOUCH_PTS',
  'PAINT_TOUCH_FG_PCT',
  'POST_TOUCH_PTS',
  'POST_TOUCH_FG_PCT',
  'ELBOW_TOUCH_PTS',
  'ELBOW_TOUCH_FG_PCT'],
 'elbowtouch_stats': ['TOUCHES',
  'ELBOW_TOUCHES',
  'ELBOW_TOUCH_FGM',
  'ELBOW_TOUCH_FGA',
  'ELBOW_TOUCH_FG_PCT',
  'ELBOW_TOUCH_FTM',
  'ELBOW_TOUCH_FTA',
  'ELBOW_TOUCH_FT_PCT',
  'ELBOW_TOUCH_PTS',
  'ELBOW_TOUCH_PTS_PCT',
  'ELBOW_TOUCH_PASSES',
  'ELBOW_TOUCH_PASSES_PCT',
  'ELBOW_TOUCH_AST',
  'ELBOW_TOUCH_AST_PCT',
  'ELBOW_TOUCH_TOV',
  'ELBOW_TOUCH_TOV_PCT',
  'ELBOW_TOUCH_FOULS',
  'ELBOW_TOUCH_FOULS_PCT'],
 'painttouch_stats': ['TOUCHES',
  'PAINT_TOUCHES',
  'PAINT_TOUCH_FGM',
  'PAINT_TOUCH_FGA',
  'PAINT_TOUCH_FG_PCT',
  'PAINT_TOUCH_FTM',
  'PAINT_TOUCH_FTA',
  'PAINT_TOUCH_FT_PCT',
  'PAINT_TOUCH_PTS',
  'PAINT_TOUCH_PTS_PCT',
  'PAINT_TOUCH_PASSES',
  'PAINT_TOUCH_PASSES_PCT',
  'PAINT_TOUCH_AST',
  'PAINT_TOUCH_AST_PCT',
  'PAINT_TOUCH_TOV',
  'PAINT_TOUCH_TOV_PCT',
  'PAINT_TOUCH_FOULS',
  'PAINT_TOUCH_FOULS_PCT

In [20]:
feature_finder("tov")

{'advanced_stats': ['TM_TOV_PCT',
  'E_TOV_PCT',
  'TM_TOV_PCT_RANK',
  'E_TOV_PCT_RANK'],
 'base_stats': ['TOV', 'TOV_RANK'],
 'drives_stats': ['DRIVE_TOV', 'DRIVE_TOV_PCT'],
 'elbowtouch_stats': ['ELBOW_TOUCH_TOV', 'ELBOW_TOUCH_TOV_PCT'],
 'painttouch_stats': ['PAINT_TOUCH_TOV', 'PAINT_TOUCH_TOV_PCT'],
 'posttouch_stats': ['POST_TOUCH_TOV', 'POST_TOUCH_TOV_PCT'],
 'scoring_splits': ['PCT_PTS_OFF_TOV', 'PCT_PTS_OFF_TOV_RANK'],
 'usage_stats': ['PCT_TOV', 'PCT_TOV_RANK']}

In [ ]:
for name, df in datasets.items():
    print(f"{name}:")
    print(datasets[name].isna().mean().sort_values(ascending=False))